In the [previous post](2025-12-23-sam3-gemini-segmentation.html), we explored SAM3's image segmentation. Now let's use SAM3 for **video tracking** - following a tennis ball across frames using just a text prompt.

## Setup

In [ ]:
# Install SAM3 from source
# !git clone https://github.com/facebookresearch/sam3.git
# !cd sam3 && pip install -e .
# !pip install supervision

In [ ]:
import os
import cv2
import torch
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
import supervision as sv
from sam3.model_builder import build_sam3_video_predictor

%config InlineBackend.figure_format = 'retina'

In [ ]:
# Initialize SAM3 video predictor
if torch.cuda.is_available():
    gpus = list(range(torch.cuda.device_count()))
else:
    gpus = []  # CPU

predictor = build_sam3_video_predictor(gpus_to_use=gpus)

## Load Video

In [ ]:
video_path = "992695-hd_1920_1080_25fps.mp4"
print(f"Video: {video_path}")

In [ ]:
# Show the original video
from IPython.display import Video
Video(video_path, embed=True, width=640)

In [ ]:
# Load video frames (first 3 seconds)
cap = cv2.VideoCapture(video_path)
fps = cap.get(cv2.CAP_PROP_FPS)
max_frames = int(fps * 3)  # 3 seconds

video_frames = []
while len(video_frames) < max_frames:
    ret, frame = cap.read()
    if not ret:
        break
    video_frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
cap.release()

print(f"Loaded {len(video_frames)} frames ({len(video_frames)/fps:.1f}s at {fps} fps)")

## Track with Text Prompt

Track "tennis ball" across all frames using just text.

In [ ]:
# Save frames temporarily for SAM3
temp_dir = Path("temp_frames")
temp_dir.mkdir(exist_ok=True)

for i, frame in enumerate(video_frames):
    Image.fromarray(frame).save(temp_dir / f"{i:05d}.jpg")

print(f"Saved {len(video_frames)} frames to {temp_dir}")

In [ ]:
# Start inference session
response = predictor.handle_request(
    request=dict(
        type="start_session",
        resource_path=str(temp_dir),
    )
)
session_id = response["session_id"]
print(f"Session started: {session_id}")

In [ ]:
# Add text prompt for "tennis ball"
response = predictor.handle_request(
    request=dict(
        type="add_prompt",
        session_id=session_id,
        frame_index=0,
        text="tennis ball",
    )
)
print(f"Prompt added, objects detected: {len(response.get('outputs', {}))}")

In [ ]:
# Propagate through video
outputs_per_frame = {}

for response in predictor.handle_stream_request(
    request=dict(
        type="propagate_in_video",
        session_id=session_id,
    )
):
    outputs_per_frame[response["frame_index"]] = response["outputs"]

print(f"Tracked {len(outputs_per_frame)} frames")

## Visualize with Supervision

In [ ]:
# Show tracking on key frames
key_frames = [0, len(video_frames)//4, len(video_frames)//2, 3*len(video_frames)//4, len(video_frames)-1]
key_frames = [f for f in key_frames if f in outputs_per_frame]

mask_annotator = sv.MaskAnnotator(color=sv.Color.GREEN, opacity=0.5)

fig, axes = plt.subplots(1, len(key_frames), figsize=(4*len(key_frames), 4))

for i, frame_idx in enumerate(key_frames):
    frame = video_frames[frame_idx].copy()
    output = outputs_per_frame.get(frame_idx, {})
    
    # Extract masks from output
    masks_list = []
    for obj_id, obj_data in output.items():
        if 'mask' in obj_data:
            masks_list.append(obj_data['mask'])
    
    if masks_list:
        masks = np.stack(masks_list)
        detections = sv.Detections(xyxy=sv.mask_to_xyxy(masks), mask=masks)
        frame = mask_annotator.annotate(frame, detections)
    
    axes[i].imshow(frame)
    axes[i].set_title(f"Frame {frame_idx}")
    axes[i].axis('off')

plt.suptitle("Tracking 'tennis ball' with SAM3", fontweight='bold')
plt.tight_layout()
plt.show()

## Save Tracked Video

In [ ]:
# Create annotated frames
annotated_frames = []

for frame_idx, frame in enumerate(video_frames):
    frame = frame.copy()
    output = outputs_per_frame.get(frame_idx, {})
    
    masks_list = []
    for obj_id, obj_data in output.items():
        if 'mask' in obj_data:
            masks_list.append(obj_data['mask'])
    
    if masks_list:
        masks = np.stack(masks_list)
        detections = sv.Detections(xyxy=sv.mask_to_xyxy(masks), mask=masks)
        frame = mask_annotator.annotate(frame, detections)
    
    annotated_frames.append(frame)

print(f"Annotated {len(annotated_frames)} frames")

In [ ]:
# Save video
output_path = "tennis_tracked.mp4"
h, w = video_frames[0].shape[:2]
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(output_path, fourcc, fps, (w, h))

for frame in annotated_frames:
    out.write(cv2.cvtColor(frame, cv2.COLOR_RGB2BGR))

out.release()
print(f"Saved: {output_path}")

In [ ]:
Video(output_path, embed=True, width=640)

In [ ]:
# Cleanup
predictor.handle_request(
    request=dict(type="close_session", session_id=session_id)
)
predictor.shutdown()

# Remove temp frames
import shutil
shutil.rmtree(temp_dir)

## Summary

SAM3's video mode enables:
- **Text-based tracking**: Just describe what to track ("tennis ball")
- **Frame propagation**: Automatically follows object across frames
- **Easy visualization**: Using [supervision](https://github.com/roboflow/supervision) for clean annotations

## References

- [Previous post: SAM3 Image Segmentation](2025-12-23-sam3-gemini-segmentation.html)
- [SAM3 GitHub](https://github.com/facebookresearch/sam3)
- [Supervision by Roboflow](https://github.com/roboflow/supervision)